# Neighbourhood Comparison MIA Adapted to Federated LLM Fine-Tuning

This notebook adapts the **neighbourhood comparison** membership inference attack (Mattern et al., *Membership Inference Attacks against Language Models via Neighbourhood Comparison*, Findings of the ACL 2023 / arXiv:2305.18462) to federated learning (FL) based fine-tuning of a causal language model.

**Original attack.** For a target text `x`, generate `n` near-identical *neighbour* texts via masked-LM (BERT) single-word substitutions, then compare `x`'s loss under the target model to the **mean** loss of its neighbours. Because the neighbours are practically interchangeable with `x`, a target loss substantially below its neighbours' mean can only come from overfitting — i.e. membership (Eq. 3: predict member when `L(x) - mean_i L(x_tilde_i) < gamma`). The neighbours replace the reference model used in LiRA, so **no reference model and no knowledge of the training distribution are required**. Best config: `n = 100` neighbours, `m = 1` word swap.

**FL adaptation.** Construct paired positive and negative worlds where a target record is either included in a target client's local fine-tuning data or absent. Federated fine-tuning (FedAvg) runs first. The server-side attacker then scores the target record under the **final federated model only**: it generates neighbours of the record, computes the model's loss on the record and on each neighbour, and takes `membership_score = mean_neighbour_loss - target_loss`. The score is thresholded, TPR/TNR/Adv measured, and results persisted in Firebase Firestore.

The attack was **designed in the fine-tuning setting**, so this is a direct transfer to a federated fine-tuned model. Its decisive advantage over reference-based FL adaptations is that the synthetic neighbours replace the reference model — nothing about the private federated training distribution needs to be known.

## Configuration

Default model: `sshleifer/tiny-gpt2`, a small open-source causal LM suitable for local smoke-scale fine-tuning. Increase the model, data size, rounds, and trials only after the smoke run passes.

The decision `threshold` is on the membership score `mean_neighbour_loss - target_loss`. A memorized record has a target loss well below its neighbours, giving a large positive score; a non-member sits near zero. For a real run, calibrate the threshold on a held-out split or report threshold-free ROC-AUC instead. `num_neighbours` defaults to a small value for the smoke run (the paper uses 100); `neighbour_swaps` is the number of word replacements per neighbour (`m = 1` in the paper).

## GPU Selection

On a shared multi-GPU box, pin this notebook to a single GPU **before** any CUDA
initialization to avoid out-of-memory crashes from other jobs. This cell must run
first (top to bottom). It picks the GPU as follows:

1. `EXPERIMENT_GPU` from the shell environment, else from `.env` (e.g. `EXPERIMENT_GPU=1`).
2. Otherwise auto-selects the GPU with the most free memory (via `nvidia-smi`).

The chosen physical GPU is exposed to this process (and to the Flower/Ray
simulation workers) as `cuda:0`. Set `EXPERIMENT_GPU=cpu` to force CPU.

In [ ]:
# Pin to one GPU before torch initializes CUDA (avoids OOM on a shared box).
import os
import shutil
import subprocess
from pathlib import Path


def _read_env_file_var(name: str):
    # Minimal .env reader so GPU pinning works before python-dotenv is loaded.
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / ".env"
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                if key.strip() == name:
                    return value.strip().strip('"').strip("'")
            break
    return None


def _gpu_free_memory():
    if shutil.which("nvidia-smi") is None:
        return []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
            text=True,
        )
    except Exception:
        return []
    rows = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        idx, free = line.split(",")
        rows.append((idx.strip(), int(free)))
    return rows


def select_gpu() -> str | None:
    forced = os.environ.get("EXPERIMENT_GPU") or _read_env_file_var("EXPERIMENT_GPU")
    if forced is not None and forced.strip() != "":
        return forced.strip()
    rows = _gpu_free_memory()
    if not rows:
        return None
    return max(rows, key=lambda r: r[1])[0]


_gpu = select_gpu()
if _gpu is None:
    print("GPU selection: no GPU detected; using default device visibility.")
elif _gpu.lower() == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("GPU selection: forced CPU (CUDA_VISIBLE_DEVICES='').")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = _gpu
    _free = dict(_gpu_free_memory()).get(_gpu)
    _detail = f" ({_free} MiB free)" if _free is not None else ""
    print(f"GPU selection: pinned to physical GPU {_gpu}{_detail}; it appears as cuda:0 in this process.")

In [ ]:
from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from itertools import product
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence
import json
import math
import os
import random
import shutil
import time

@dataclass(frozen=True)
class ExperimentConfig:
    attack_name: str = "neighborhood"
    paper_source: str = "Mattern et al. 2023 (arXiv:2305.18462) Neighbourhood Comparison MIA"
    model_id: str = "sshleifer/tiny-gpt2"
    dataset_name: str = "synthetic_client_text"
    num_clients: int = 4
    clients_per_round: int = 4
    federated_rounds: int = 1
    local_epochs: int = 1
    local_batch_size: int = 2
    client_lr: float = 5e-5
    target_client_id: int = 0
    attack_trials: int = 4
    # Neighbour generation: paper uses n=100, m=1. Smaller default here for the smoke run.
    num_neighbours: int = 25   # paper best config: 100
    neighbour_swaps: int = 1   # m: single-word replacement (paper best config)
    # Threshold on the membership score = mean_neighbour_loss - target_loss.
    # Positive => target loss below neighbours (memorized). Members should
    # exceed this; non-members sit near zero.
    # Threshold on the membership score = mean_neighbour_loss - target_loss.
    # Members sit clearly below their neighbours' mean loss, so their score is
    # larger. Calibrated to separate the toy scores; calibrate on a held-out
    # split (or report threshold-free ROC-AUC) for a real run.
    threshold: float = 0.02
    max_length: int = 64
    seed: int = 7
    firestore_collection: str = "ami_federated_llm_results"
    artifact_root: str = "artifacts/neighborhood_adaptation"
    fl_framework: str = "flower"
    sim_num_gpus: float = 0.0
    keep_artifacts: bool = False
    use_hf_models: bool = False  # Set True for a real Hugging Face fine-tuning run.

BASE_CONFIG = ExperimentConfig()

SWEEP = {
    "model_id": ["sshleifer/tiny-gpt2"],
    "federated_rounds": [1],
    "num_clients": [4],
    "local_epochs": [1],
    "client_lr": [5e-5],
    "num_neighbours": [25],
    "seed": [7],
}


def expand_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    keys = list(sweep.keys())
    for values in product(*(sweep[key] for key in keys)):
        yield replace(base_config, **dict(zip(keys, values)))


def experiment_key(config: ExperimentConfig) -> str:
    payload = json.dumps(asdict(config), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()[:16]


def artifact_dir_for(config: ExperimentConfig) -> Path:
    return Path(config.artifact_root) / experiment_key(config)

In [ ]:
# Load credentials from a local .env file (see .env.example).
# The notebooks read os.environ directly, so this must run before any
# Firestore or Hugging Face call.
try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    %pip install -q python-dotenv
    from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# huggingface_hub / transformers automatically read HF_TOKEN from the
# environment for model downloads and gated/private repos.
print("Credentials loaded:", {
    "FIREBASE_SERVICE_ACCOUNT_JSON": bool(os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")),
    "GOOGLE_APPLICATION_CREDENTIALS": bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")),
    "FIREBASE_PROJECT_ID": bool(os.environ.get("FIREBASE_PROJECT_ID")),
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN")),
})

## Firestore Cache Check

The cache is checked before any fine-tuning. If no Firebase credentials are present, the notebook can still run locally and returns uncached results.

In [ ]:
def get_firestore_client(project_id: Optional[str] = None):
    import firebase_admin
    from firebase_admin import credentials, firestore

    if not firebase_admin._apps:
        raw_json = os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")
        cred_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        if raw_json:
            cred = credentials.Certificate(json.loads(raw_json))
        elif cred_path:
            cred = credentials.Certificate(cred_path)
        else:
            raise RuntimeError("Set FIREBASE_SERVICE_ACCOUNT_JSON or GOOGLE_APPLICATION_CREDENTIALS.")

        options = {"projectId": project_id} if project_id else None
        firebase_admin.initialize_app(cred, options=options)

    return firestore.client()


def load_cached_result(config: ExperimentConfig):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception as exc:
        return None
    snapshot = db.collection(config.firestore_collection).document(experiment_key(config)).get()
    if snapshot.exists:
        payload = snapshot.to_dict()
        if payload.get("status") == "complete":
            return payload
    return None

## Federated Fine-Tuning

The positive and negative worlds differ only in whether the target client's local data contains the target record. The FL control flow is FedAvg: copy global weights to selected clients, train locally, collect client weights, and average them into the next global model.

The membership signal is entirely in the **model's loss on the target record relative to its neighbours**. The neighbours are text perturbations of the target record and do not depend on the model, so they are identical across the positive and negative worlds — the model trained *with* the target record should assign it a lower loss than its neighbours (a member signal), while the model trained *without* it should not.

In [ ]:
TARGET_RECORD = "Client 0 private appointment note: Ana's insulin refill is scheduled for Friday at 10am."
HELD_OUT_RECORD = "Public clinic reminder: bring your insurance card and arrive ten minutes early."
CLIENT_CORPUS = [
    ["Client 0 billing question about invoice dates.", "Client 0 support chat about portal login."],
    ["Client 1 shipping update for a replacement device.", "Client 1 warranty call summary."],
    ["Client 2 product feedback about keyboard layout.", "Client 2 short troubleshooting note."],
    ["Client 3 scheduling request for a generic follow-up.", "Client 3 public FAQ paraphrase."],
]


def build_client_partitions(config: ExperimentConfig, truth_member: bool) -> List[List[str]]:
    random.seed(config.seed)
    partitions = [list(records) for records in CLIENT_CORPUS[: config.num_clients]]
    while len(partitions) < config.num_clients:
        partitions.append([f"Synthetic client {len(partitions)} ordinary support record."])
    target_payload = TARGET_RECORD if truth_member else HELD_OUT_RECORD
    partitions[config.target_client_id].append(target_payload)
    return partitions


class ToyFederatedLM:
    """Dependency-light smoke model that mimics memorization by token-count updates."""

    def __init__(self):
        self.token_counts = {}

    def copy(self):
        clone = ToyFederatedLM()
        clone.token_counts = dict(self.token_counts)
        return clone

    def fit(self, texts: Sequence[str], epochs: int = 1):
        for _ in range(epochs):
            for text in texts:
                for token in text.lower().split():
                    self.token_counts[token] = self.token_counts.get(token, 0.0) + 1.0
        return self

    def nll(self, text: str) -> float:
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        total = sum(self.token_counts.values()) + 1.0
        vocab = len(self.token_counts) + 1.0
        score = 0.0
        for token in tokens:
            prob = (self.token_counts.get(token, 0.0) + 1.0) / (total + vocab)
            score += -math.log(prob)
        return score / len(tokens)


def toy_fedavg(global_model: ToyFederatedLM, client_models: Sequence[ToyFederatedLM]) -> ToyFederatedLM:
    merged = ToyFederatedLM()
    keys = set().union(*(model.token_counts.keys() for model in client_models)) if client_models else set()
    for key in keys:
        merged.token_counts[key] = sum(model.token_counts.get(key, 0.0) for model in client_models) / len(client_models)
    return merged


def run_toy_federated_finetune(config: ExperimentConfig, truth_member: bool):
    global_model = ToyFederatedLM()
    history = []
    for round_id in range(config.federated_rounds):
        partitions = build_client_partitions(config, truth_member=truth_member)
        selected = list(range(min(config.clients_per_round, len(partitions))))
        client_models = []
        for client_id in selected:
            local_model = global_model.copy().fit(partitions[client_id], epochs=config.local_epochs)
            client_models.append(local_model)
        global_model = toy_fedavg(global_model, client_models)
        history.append({"round": round_id, "selected_clients": selected})
    return global_model, history

### Hugging Face FL Hook

Set `use_hf_models=True` after installing `torch`, `transformers`, and `"flwr[simulation]"`. This hook keeps the same positive/negative world construction as the smoke model, but performs genuine federated fine-tuning with the official Flower framework: each client is a `flwr.client.NumPyClient` running `AutoModelForCausalLM`, aggregated by the built-in `FedAvg` strategy through `flwr.simulation.run_simulation`. The dependency-light `ToyFederatedLM` smoke path above is unchanged. The neighbourhood attack needs only this single fine-tuned model — the BERT neighbour generator (loaded in the attack cell) replaces the reference model.

In [ ]:
def run_hf_federated_finetune(config: ExperimentConfig, truth_member: bool):
    """Genuine federated fine-tuning of an open-source causal LM with Flower (flwr).

    Each client is a NumPyClient that locally fine-tunes the model on its
    partition; the server runs the FedAvg strategy through
    flwr.simulation.run_simulation. Every client reports num_examples=1 so
    FedAvg's example-weighted average reduces to a plain unweighted mean.
    """
    from collections import OrderedDict

    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import flwr
    from flwr.client import NumPyClient, ClientApp
    from flwr.common import Context, ndarrays_to_parameters, parameters_to_ndarrays
    from flwr.server import ServerApp, ServerAppComponents, ServerConfig
    from flwr.server.strategy import FedAvg
    from flwr.simulation import run_simulation

    use_cuda = config.sim_num_gpus > 0 and torch.cuda.is_available()
    client_dev = "cuda" if use_cuda else "cpu"
    eval_dev = "cuda" if torch.cuda.is_available() else "cpu"

    partitions = build_client_partitions(config, truth_member=truth_member)
    num_clients = len(partitions)

    def get_parameters(model):
        return [value.detach().cpu().numpy() for value in model.state_dict().values()]

    def set_parameters(model, parameters):
        state_dict = OrderedDict(
            (key, torch.tensor(value)) for key, value in zip(model.state_dict().keys(), parameters)
        )
        model.load_state_dict(state_dict, strict=True)

    def load_model_and_tokenizer():
        tokenizer = AutoTokenizer.from_pretrained(config.model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(config.model_id)
        return model, tokenizer

    class NeighborhoodFlowerClient(NumPyClient):
        def __init__(self, partition_id, texts):
            self.partition_id = partition_id
            self.texts = texts

        def fit(self, parameters, fit_config):
            model, tokenizer = load_model_and_tokenizer()
            set_parameters(model, parameters)
            model.to(client_dev)
            model.train()
            encoded = tokenizer(
                self.texts,
                padding=True,
                truncation=True,
                max_length=config.max_length,
                return_tensors="pt",
            )
            dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
            loader = DataLoader(dataset, batch_size=config.local_batch_size, shuffle=True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=config.client_lr)
            for _ in range(config.local_epochs):
                for input_ids, attention_mask in loader:
                    input_ids = input_ids.to(client_dev)
                    attention_mask = attention_mask.to(client_dev)
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                    outputs.loss.backward()
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            updated = get_parameters(model)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # num_examples=1 -> FedAvg weighted mean reduces to an unweighted mean.
            return updated, 1, {"partition_id": self.partition_id}

    init_model, _ = load_model_and_tokenizer()
    initial_parameters = ndarrays_to_parameters(get_parameters(init_model))
    del init_model

    clients_per_round = min(config.clients_per_round, num_clients)
    fraction_fit = clients_per_round / num_clients
    capture = {"parameters": None, "history": []}

    class SaveModelFedAvg(FedAvg):
        def aggregate_fit(self, server_round, results, failures):
            if results:
                selected = [int(fitres.metrics.get("partition_id", -1)) for _, fitres in results]
                capture["history"].append({"round": server_round - 1, "selected_clients": selected})
            aggregated_parameters, aggregated_metrics = super().aggregate_fit(server_round, results, failures)
            if aggregated_parameters is not None:
                capture["parameters"] = parameters_to_ndarrays(aggregated_parameters)
            return aggregated_parameters, aggregated_metrics

    def client_fn(context: Context):
        partition_id = int(context.node_config["partition-id"])
        return NeighborhoodFlowerClient(partition_id, partitions[partition_id]).to_client()

    def server_fn(context: Context):
        strategy = SaveModelFedAvg(
            fraction_fit=fraction_fit,
            fraction_evaluate=0.0,
            min_fit_clients=clients_per_round,
            min_available_clients=num_clients,
            initial_parameters=initial_parameters,
        )
        return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=config.federated_rounds))

    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": float(config.sim_num_gpus)}}
    run_simulation(
        server_app=ServerApp(server_fn=server_fn),
        client_app=ClientApp(client_fn=client_fn),
        num_supernodes=num_clients,
        backend_config=backend_config,
    )

    global_model, tokenizer = load_model_and_tokenizer()
    if capture["parameters"] is not None:
        set_parameters(global_model, capture["parameters"])
    global_model.to(eval_dev).eval()
    return {"model": global_model, "tokenizer": tokenizer, "device": eval_dev}, capture["history"]

## Attack Construction

The attacker computes the **final FL model's** mean per-token loss on the target record, generates `num_neighbours` near-identical neighbours (single-word `m = 1` substitutions), computes the model's loss on each neighbour, and forms `membership_score = mean_neighbour_loss - target_loss`. Unlike the reference-model attack, there is **no second target-domain model**: the neighbours supply the calibration. A memorized record receives an unusually low loss relative to its interchangeable neighbours, so the mean neighbour loss exceeds the target loss and the score is large and positive.

- **Toy/smoke path.** Neighbours are synthesized by trivial single-token perturbations of the target record (swap one word for a benign filler token). The `ToyFederatedLM` trained *with* the target record has boosted counts for the record's exact tokens, so it scores the record lower (lower NLL) than its perturbed neighbours — a positive membership score. Trained *without* the record, no such gap appears.
- **HF path.** Neighbours are generated with a real `bert-base-uncased` masked LM inside the guarded HF branch, following the paper's Algorithm 1 (dropout on the original token embedding, `m = 1`).

In [ ]:
def neighborhood_membership_score(target_loss: float, neighbour_losses: Sequence[float]) -> float:
    """Membership score = mean(neighbour_losses) - target_loss. Higher => member.

    Negation of the paper's Eq. 3 quantity so that members (target loss well
    below the neighbour mean) score higher, matching the >= threshold convention.
    """
    if not neighbour_losses:
        raise ValueError("Need at least one neighbour loss to calibrate.")
    return (sum(neighbour_losses) / len(neighbour_losses)) - target_loss


# Benign filler tokens used to build toy single-word (m=1) neighbours.
_TOY_FILLER_TOKENS = [
    "today", "here", "please", "note", "update", "kindly", "again", "soon",
    "however", "meanwhile", "generally", "actually", "somewhat", "briefly",
    "recently", "currently", "possibly", "perhaps", "namely", "therefore",
    "additionally", "otherwise", "regardless", "accordingly", "nonetheless",
]


def generate_neighbours_toy(text: str, n: int, swaps: int = 1, seed: int = 0) -> List[str]:
    """Synthesize n neighbours by replacing `swaps` word(s) with benign fillers.

    Each neighbour differs from the target by a single-word substitution (m=1
    by default), mirroring the paper's best configuration. The perturbations
    are semantics-preserving-ish fillers, and — importantly — they are NOT the
    exact target string, so a model that memorized the target scores it lower
    than these neighbours.
    """
    rng = random.Random(seed)
    words = text.split()
    if not words:
        return [text]
    neighbours = []
    for i in range(n):
        variant = list(words)
        for _ in range(max(1, swaps)):
            pos = rng.randrange(len(variant))
            variant[pos] = _TOY_FILLER_TOKENS[(i + pos) % len(_TOY_FILLER_TOKENS)]
        neighbours.append(" ".join(variant))
    return neighbours


def score_candidate_toy(target_model: ToyFederatedLM, text: str, config: ExperimentConfig) -> float:
    neighbours = generate_neighbours_toy(
        text, n=config.num_neighbours, swaps=config.neighbour_swaps, seed=config.seed
    )
    target_loss = target_model.nll(text)
    neighbour_losses = [target_model.nll(nb) for nb in neighbours]
    return neighborhood_membership_score(target_loss, neighbour_losses)


def generate_neighbours_bert(text, tokenizer_mlm, model_mlm, n=100, dropout_p=0.7, device="cpu", max_length=128):
    """Paper-faithful single-word (m=1) neighbour generation (Algorithm 1).

    Strong dropout is applied to the ORIGINAL token embedding (the token is not
    masked out) so BERT respects the original word's meaning; candidates are
    ranked by the normalised suitability p_swap = p(w_tilde) / (1 - p(original)).
    """
    import torch

    encoded = tokenizer_mlm(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = encoded["input_ids"].to(device)
    ids = input_ids[0]
    special = set(tokenizer_mlm.all_special_ids)
    embedding_layer = model_mlm.get_input_embeddings()
    candidates = []
    for pos in range(ids.shape[0]):
        original_id = int(ids[pos].item())
        if original_id in special:
            continue
        base_embeds = embedding_layer(input_ids)
        dropped = torch.nn.functional.dropout(base_embeds[:, pos, :], p=dropout_p, training=True)
        perturbed = base_embeds.clone()
        perturbed[:, pos, :] = dropped
        with torch.no_grad():
            logits = model_mlm(inputs_embeds=perturbed).logits
        probs = torch.softmax(logits[0, pos], dim=-1)
        denom = max(1e-8, 1.0 - float(probs[original_id].item()))
        topk = torch.topk(probs, k=min(10, probs.shape[-1]))
        for score, cand_id in zip(topk.values.tolist(), topk.indices.tolist()):
            if cand_id == original_id or cand_id in special:
                continue
            candidates.append((score / denom, pos, cand_id))
    candidates.sort(key=lambda c: c[0], reverse=True)
    neighbours = []
    for _, pos, cand_id in candidates[:n]:
        new_ids = ids.clone()
        new_ids[pos] = cand_id
        neighbours.append(tokenizer_mlm.decode(new_ids, skip_special_tokens=True))
    return neighbours


def _mean_token_nll_hf(model, tokenizer, text, device, max_length):
    import torch
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    if encoded["input_ids"].shape[-1] < 2:
        return 0.0
    with torch.no_grad():
        outputs = model(**encoded, labels=encoded["input_ids"])
    return float(outputs.loss.detach().cpu())


def score_candidate_hf(target_bundle, text: str, config: ExperimentConfig) -> float:
    """Generate BERT neighbours and score the record + neighbours under the FL model."""
    from transformers import AutoModelForMaskedLM, AutoTokenizer

    model = target_bundle["model"]
    tokenizer = target_bundle["tokenizer"]
    device = target_bundle["device"]

    mlm_name = "bert-base-uncased"
    mlm_tokenizer = AutoTokenizer.from_pretrained(mlm_name)
    mlm_model = AutoModelForMaskedLM.from_pretrained(mlm_name).to(device).eval()

    neighbours = generate_neighbours_bert(
        text, mlm_tokenizer, mlm_model, n=config.num_neighbours,
        device=device, max_length=config.max_length,
    )
    target_loss = _mean_token_nll_hf(model, tokenizer, text, device, config.max_length)
    neighbour_losses = [
        _mean_token_nll_hf(model, tokenizer, nb, device, config.max_length) for nb in neighbours
    ]
    return neighborhood_membership_score(target_loss, neighbour_losses)

## Attack Execution

Each trial samples a positive or negative world, runs FL fine-tuning, then attacks the target record with the neighbourhood comparison. This keeps the target membership at the client-data level rather than collapsing the experiment into centralized fine-tuning.

In [ ]:
def run_attack_trial(config: ExperimentConfig, trial_id: int, truth_member: bool):
    trial_config = replace(config, seed=config.seed + trial_id)
    if trial_config.use_hf_models:
        target_bundle, history = run_hf_federated_finetune(trial_config, truth_member=truth_member)
        score = score_candidate_hf(target_bundle, TARGET_RECORD, trial_config)
    else:
        target_model, history = run_toy_federated_finetune(trial_config, truth_member=truth_member)
        score = score_candidate_toy(target_model, TARGET_RECORD, trial_config)

    pred_member = score >= trial_config.threshold
    return {
        "trial_id": trial_id,
        "truth_member": bool(truth_member),
        "score": float(score),
        "pred_member": bool(pred_member),
        "federated_history": history,
    }


def run_attack_trials(config: ExperimentConfig):
    trials = []
    for trial_id in range(config.attack_trials):
        truth_member = (trial_id % 2 == 0)
        trials.append(run_attack_trial(config, trial_id=trial_id, truth_member=truth_member))
    return trials

## Measurement

The primary metric follows the FL AMI project convention: `Adv = 0.5 * TPR + 0.5 * TNR`. A threshold-free `roc_auc` is included because the neighbourhood score's absolute scale depends on the model and dataset and a fixed threshold is brittle; AUC summarizes ranking quality independent of the threshold. Other secondary diagnostics are included for auditability.

In [ ]:
def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def summarize_trials(trials: Sequence[Dict]):
    tp = sum(1 for row in trials if row["truth_member"] and row["pred_member"])
    tn = sum(1 for row in trials if not row["truth_member"] and not row["pred_member"])
    fp = sum(1 for row in trials if not row["truth_member"] and row["pred_member"])
    fn = sum(1 for row in trials if row["truth_member"] and not row["pred_member"])
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    labels = [row["truth_member"] for row in trials]
    scores = [row["score"] for row in trials]
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(trials) if trials else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, scores),
        "num_trials": len(trials),
    }

## Firestore Write and Cleanup

Firestore stores compact configuration, metrics, trial records, history, status, timestamps, and artifact references. Large model weights should be stored locally or in Cloud Storage, with only paths or `gs://` URIs written to Firestore.

Two Firestore-specific guards are applied: per-trial round histories (lists) are wrapped inside maps so the document contains **no directly nested arrays** (Firestore rejects those), and `save_result` only swallows the *missing-credentials* case — a genuine serialization/write error is re-raised so a broken document shape fails fast on the smoke run instead of silently looking "not saved".

In [ ]:
def save_result(config: ExperimentConfig, result: Dict):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception:
        # Missing-credentials case only: it is fine to skip writing locally.
        return False
    # A real write/serialization error (e.g. nested-array rejection) propagates
    # so it fails fast rather than masquerading as "not saved".
    db.collection(config.firestore_collection).document(experiment_key(config)).set(result, merge=True)
    return True


def cleanup_artifacts(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)


def run_single_experiment(config: ExperimentConfig):
    run_id = experiment_key(config)
    cached = load_cached_result(config)
    if cached and cached.get("status") == "complete":
        return cached

    artifact_dir = artifact_dir_for(config)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    trials = run_attack_trials(config)
    metrics = summarize_trials(trials)
    result = {
        "run_id": run_id,
        "status": "complete",
        "updated_at_unix": int(time.time()),
        "config": asdict(config),
        "methodology": {
            "paper_attack": "Neighbourhood comparison MIA: for target x, generate n near-identical neighbours via BERT masked-LM single-word (m=1) substitution, then predict member when L(x) is substantially below the mean neighbour loss (Eq. 3). Reference-model-free: neighbours replace the reference model.",
            "llm_adaptation": "Positive and negative FL worlds differ by target-client membership; after Flower (flwr) FedAvg simulation, score the target record under the final FL model only by generating neighbours and taking mean_neighbour_loss - target_loss. No reference model is used.",
            "metric_definition": "Adv = 0.5 * TPR + 0.5 * TNR; membership score = mean_neighbour_loss - target_loss so members (loss below neighbours) score higher.",
            "deviation_from_source": "The attack was designed for the fine-tuning setting, so applying it to a federated fine-tuned model is a direct transfer. No reference model is needed: the neighbours are the difficulty calibration. The smoke run uses a deterministic toy causal scorer and toy single-word neighbour perturbations; set use_hf_models=True for genuine federated fine-tuning of an open-source LLM (Flower FedAvg) with BERT (bert-base-uncased) neighbour generation following Algorithm 1 (n=100, m=1, dropout p=0.7 on the original embedding).",
        },
        # Firestore forbids directly nested arrays, so wrap each trial's
        # per-round history (itself a list) inside a map.
        "federated_history": [
            {"trial_id": row["trial_id"], "rounds": row["federated_history"]}
            for row in trials
        ],
        "metrics": metrics,
        "attack_trials": [
            {key: row[key] for key in ("trial_id", "truth_member", "score", "pred_member")}
            for row in trials
        ],
        "artifacts": {
            "artifact_dir": str(artifact_dir),
            "federated_model_path": None,
            "reference_model": None,  # Neighbourhood attack uses no reference model.
        },
    }
    saved = save_result(config, result)
    result["firestore_saved"] = saved
    if saved and not config.keep_artifacts:
        cleanup_artifacts(artifact_dir)
    return result


def run_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    return [run_single_experiment(config) for config in expand_sweep(base_config, sweep)]

## Smoke Run

This smoke run verifies the full ordering: Firestore cache check, FL fine-tuning, adapted neighbourhood attack execution, measurement, optional Firestore write, and artifact cleanup. It does not download a model. For full reproduction, set `BASE_CONFIG = replace(BASE_CONFIG, use_hf_models=True)` and ensure Firebase credentials are configured.

The smoke run also validates the Firestore document shape (the `federated_history` is a list of maps, not nested arrays) before any long fine-tuning job, so a persistence error cannot waste a real run.

In [ ]:
smoke_config = replace(BASE_CONFIG, attack_trials=4, use_hf_models=False)
smoke_result = run_single_experiment(smoke_config)

assert smoke_result["metrics"]["num_trials"] == 4, smoke_result
assert any(row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
assert any(not row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
for key in ("tpr", "tnr", "adv", "roc_auc"):
    assert key in smoke_result["metrics"], smoke_result

# Neighbourhood ranking sanity: members (target loss below neighbour mean) must
# outrank non-members regardless of threshold.
assert smoke_result["metrics"]["roc_auc"] == 1.0, smoke_result

# Threshold sanity: the default threshold must actually separate the two
# worlds, not just rank them (Adv is the primary metric, per AGENTS.md).
assert smoke_result["metrics"]["adv"] == 1.0, smoke_result["attack_trials"]

# Firestore shape guard: federated_history is a list of maps (no nested arrays).
fh = smoke_result["federated_history"]
assert isinstance(fh, list) and all(isinstance(item, dict) for item in fh), fh

smoke_result